# V028: coherent quantization-error feasibility, not pressure prediction


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import csv,json,hashlib,time,math
import numpy as np
import pandas as pd

V028_RECIPE=dict(primary_seed=20262801,replicate_seed=20262802,per_RC=64,bootstrap_seed=20262803,bootstrap_draws=1000,
    family='decay_sum_constant_reference',pressure_lag=0,estimator='ordinary_least_squares',fit_fraction=.7,
    minimum_eligible=14,minimum_held=3,maximum_condition=1e10,physical_tolerance=1e-6,
    control_low=1e-6,control_high=99.999999,native_control_budget=1e-4,
    shuffle_seed=20262804,minimum_coverage=.10,minimum_increment=.05)

def q28_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def q28_code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
def q28_need(c,m):
    if not bool(c):raise ValueError(m)
def q28_json(p,v):
    with Path(p).open('x') as f:f.write(json.dumps(v,indent=2,allow_nan=False)+'\n')
def q28_arrayhash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256(str((a.shape,a.dtype.str)).encode());h.update(memoryview(a).cast('B'));return h.hexdigest()

def q28_lattice(pool_values):
    v=np.unique(np.asarray(pool_values,dtype=np.float64));q28_need(len(v)>=20 and np.isfinite(v).all(),'Insufficient finite training-pool lattice data')
    magnitude=max(1.,float(np.max(np.abs(v))));roundoff=4096*np.finfo(np.float64).eps*magnitude
    gaps=np.diff(v);gaps=gaps[gaps>64*np.finfo(np.float64).eps*magnitude]
    q28_need(len(gaps)>=10,'Insufficient resolved lattice gaps')
    nominal=float(np.median(gaps));codes=np.rint((v-v[0])/nominal)
    X=np.column_stack([codes,np.ones(len(codes))]);q,origin=np.linalg.lstsq(X,v,rcond=None)[0]
    residual=float(np.max(np.abs(v-(origin+q*codes))))
    adjacent=np.diff(codes);fraction=float(np.mean(adjacent==1))
    q28_need(q>0 and residual<=roundoff and fraction>=.5,'Uniform raw float64 training lattice not verified; do not widen bins')
    return dict(spacing=float(q),origin=float(origin),half_width=float(q/2),maximum_lattice_residual=residual,roundoff_bound=roundoff,
        observed_levels=len(v),adjacent_level_fraction=fraction,missing_levels_never_widen_cells=True,
        limitation='Empirically supported uniform lattice; neither observation order nor true latent pressure is established')

def q28_joint(r,k,A,B,h,budget=1e-4):
    """Exact interval projection/backtracking for a disjoint forest of adjacent equations.

    Each held equation is r=-A*delta[k]+B*delta[k-1]+eta,
    abs(eta)<=budget, and every delta lies in the SAME [-h,h] cell.
    Physical coefficients imply A>B>0. Independent row bounds alone are insufficient.
    """
    r=np.asarray(r,dtype=np.float64);k=np.asarray(k,dtype=np.int64)
    q28_need(r.shape==k.shape and len(k)>0 and np.all(np.diff(k)>0) and k.min()>=1 and k.max()<80,'Invalid held equations')
    q28_need(A>B>0 and h>0 and budget>0,'Physical positive A>B and finite cells required')
    delta=np.zeros(80,dtype=np.float64);interval={};chains=[];current=[]
    for j,t in enumerate(k):
        if j==0 or t!=k[j-1]+1:
            if current:chains.append(current)
            current=[];interval[int(t-1)]=(-h,h)
        lo,hi=interval[int(t-1)]
        lower=max(-h,(B*lo-r[j]-budget)/A);upper=min(h,(B*hi-r[j]+budget)/A)
        if lower>upper:return dict(feasible=False,reason='joint_interval_empty',delta=None)
        interval[int(t)]=(lower,upper);current.append(j)
    if current:chains.append(current)
    for chain in chains:
        last=int(k[chain[-1]]);lo,hi=interval[last];delta[last]=.5*(lo+hi)
        for j in reversed(chain):
            t=int(k[j]);lo,hi=interval[t-1]
            lower=max(lo,(r[j]+A*delta[t]-budget)/B);upper=min(hi,(r[j]+A*delta[t]+budget)/B)
            q28_need(lower<=upper+1e-10,'Backtracking numerical failure')
            delta[t-1]=.5*(lower+upper)
    remainder=r-(-A*delta[k]+B*delta[k-1]);violation=float(np.max(np.abs(remainder)))
    q28_need(np.max(np.abs(delta))<=h+1e-10 and violation<=budget+1e-9,'Joint certificate failed independent residual/bound check')
    return dict(feasible=True,reason='joint_certificate',delta=delta,maximum_remainder=violation,maximum_abs_delta=float(np.max(np.abs(delta))))

def q28_prefix(u,p,phase):
    k=np.arange(2,80);ok=(phase[k]==0)&(phase[k-1]==0)&(u[k]>V028_RECIPE['control_low'])&(u[k]<V028_RECIPE['control_high'])&(u[k-1]>V028_RECIPE['control_low'])&(u[k-1]<V028_RECIPE['control_high']);k=k[ok]
    if len(k)<V028_RECIPE['minimum_eligible']:return dict(status='insufficient_eligible',k=k)
    X=np.column_stack([u[k-1],-p[k],p[k-1],np.ones(len(k))]);cut=max(6,int(V028_RECIPE['fit_fraction']*len(k)))
    if len(k)-cut<V028_RECIPE['minimum_held']:return dict(status='insufficient_held',k=k)
    scale=np.maximum(np.sqrt(np.mean(X[:cut]**2,axis=0)),1e-12);Z=X[:cut]/scale
    if np.linalg.matrix_rank(Z)<4:return dict(status='rank_deficient',k=k)
    condition=float(np.linalg.cond(Z))
    if condition>=V028_RECIPE['maximum_condition']:return dict(status='ill_conditioned',k=k,condition=condition)
    coef=np.linalg.lstsq(Z,u[k[:cut]],rcond=None)[0]/scale;d,A,B,c=map(float,coef);tol=V028_RECIPE['physical_tolerance']
    kp=B/d if abs(d)>1e-12 else float('nan');ki=A-kp;reference=c/(A-B) if A-B>1e-8 else float('nan')
    physical=bool(1e-8<d<=1+tol and kp>1e-8 and ki>=-tol and A-B>1e-8 and np.isfinite(reference) and -tol<=reference<=100+tol)
    return dict(status='physical' if physical else 'physical_invalid',k=k,cut=cut,coef=coef,condition=condition,
        decay=d,kp=kp,ki=ki,reference=reference)

def q28_one(u,p,phase,h,breath_id):
    f=q28_prefix(u,p,phase);row=dict(breath_id=int(breath_id),status=f['status'],eligible_points=len(f['k']),physical=False,zero_joint=False,quantized_joint=False,shuffled_joint=False,rowwise_necessary=False)
    if 'coef' not in f:return row,None
    d,A,B,c=f['coef'];row.update(condition=f['condition'],decay=float(d),A=float(A),B=float(B),intercept=float(c),kp=float(f['kp']),ki=float(f['ki']),reference=float(f['reference']) if np.isfinite(f['reference']) else None,fit_points=f['cut'],held_points=len(f['k'])-f['cut'])
    if f['status']!='physical':return row,None
    k=f['k'][f['cut']:];r=d*u[k-1]-A*p[k]+B*p[k-1]+c-u[k];budget=V028_RECIPE['native_control_budget']
    joint=q28_joint(r,k,A,B,h,budget);coords=np.unique(np.r_[k,k-1]);coords=coords[coords>f['k'][f['cut']-1]]
    permuted=p.copy();rng=np.random.default_rng(V028_RECIPE['shuffle_seed']+int(breath_id));permuted[coords]=p[rng.permutation(coords)]
    shuffled_r=d*u[k-1]-A*permuted[k]+B*permuted[k-1]+c-u[k];shuffled=q28_joint(shuffled_r,k,A,B,h,budget)
    row.update(physical=True,zero_joint=bool(np.max(np.abs(r))<=budget),quantized_joint=joint['feasible'],shuffled_joint=shuffled['feasible'],rowwise_necessary=bool(np.all(np.abs(r)<=h*(A+B)+budget)),held_max_control_residual=float(np.max(np.abs(r))),held_median_control_residual=float(np.median(np.abs(r))),native_row_bound=float(h*(A+B)),negative_pressure_identical=bool(np.array_equal(permuted[coords],p[coords])),maximum_certificate_remainder=joint.get('maximum_remainder'))
    return row,joint['delta']

def q28_sample(rc,seed,excluded=()):
    rng=np.random.default_rng(seed);selected=[]
    for setting in sorted(set(map(tuple,rc))):
        ids=np.flatnonzero(np.all(rc==setting,axis=1));ids=np.setdiff1d(ids,np.asarray(excluded,dtype=np.int64));q28_need(len(ids)>=V028_RECIPE['per_RC'],'Too few disjoint query breaths');selected.extend(rng.choice(ids,V028_RECIPE['per_RC'],replace=False).tolist())
    return np.sort(np.asarray(selected,dtype=np.int64))

def q28_load_fit(path,fit_ids):
    bids=np.sort(np.asarray(fit_ids,dtype=np.int64));at={int(b):i for i,b in enumerate(bids)};u=np.empty((len(bids),80));p=np.empty_like(u);phase=np.empty((len(bids),80),dtype=np.int8);rc=np.empty((len(bids),2),dtype=np.int16);counts=np.zeros(len(bids),dtype=np.int16)
    with Path(path).open(newline='') as f:
        for row in csv.DictReader(f):
            i=at.get(int(row['breath_id']))
            if i is None:continue
            j=int(counts[i]);q28_need(j<80,'More than80 rows');setting=(int(row['R']),int(row['C']))
            if j==0:rc[i]=setting
            else:q28_need(tuple(rc[i])==setting,'R/C changed within breath')
            u[i,j]=float(row['u_in']);p[i,j]=float(row['pressure']);phase[i,j]=int(row['u_out']);counts[i]+=1
    q28_need(len(bids)==61114 and np.all(counts==80) and np.isfinite(u).all() and np.isfinite(p).all() and np.isin(phase,[0,1]).all(),'Afit data coverage/quality failed')
    return bids,u,p,phase,rc

def q28_gate(rows,seed):
    q=np.array([r['quantized_joint'] for r in rows],dtype=float);z=np.array([r['zero_joint'] for r in rows],dtype=float);s=np.array([r['shuffled_joint'] for r in rows],dtype=float);rng=np.random.default_rng(seed);dz=[];ds=[]
    for _ in range(V028_RECIPE['bootstrap_draws']):
        idx=rng.integers(0,len(rows),len(rows));dz.append(float((q[idx]-z[idx]).mean()));ds.append(float((q[idx]-s[idx]).mean()))
    iz=np.quantile(dz,[.025,.975]);is_=np.quantile(ds,[.025,.975]);gainz=float((q-z).mean());gains=float((q-s).mean())
    passed=bool(q.mean()>=V028_RECIPE['minimum_coverage'] and gainz>=V028_RECIPE['minimum_increment'] and gains>=V028_RECIPE['minimum_increment'] and iz[0]>0 and is_[0]>0)
    return dict(queries=len(rows),physical_count=sum(r['physical'] for r in rows),quantized_joint_count=int(q.sum()),zero_joint_count=int(z.sum()),shuffled_joint_count=int(s.sum()),rowwise_necessary_count=sum(r['rowwise_necessary'] for r in rows),quantized_joint_fraction=float(q.mean()),zero_joint_fraction=float(z.mean()),shuffled_joint_fraction=float(s.mean()),increment_over_zero=gainz,increment_over_shuffle=gains,bootstrap_increment_over_zero_95=iz.tolist(),bootstrap_increment_over_shuffle_95=is_.tolist(),gate_passed=passed)

def q28_preflight():
    from scipy.optimize import linprog
    checks={}
    def check(n,c):q28_need(c,n);checks[n]=True
    h=.035;q=.07;steps=np.arange(80);latent=10+3*np.sin(steps*.13)+.02*steps;logged=np.round(latent/q)*q;A=1.2;B=1.;d=.96;c=(A-B)*20;u=np.empty(80);u[0]=40
    for t in range(1,80):u[t]=d*u[t-1]-A*latent[t]+B*latent[t-1]+c
    k=np.arange(2,80);r=d*u[k-1]-A*logged[k]+B*logged[k-1]+c-u[k]
    check('latent_then_rounded_joint_feasible',q28_joint(r,k,A,B,h)['feasible']);check('rounded_after_controller_zero_rejected',np.max(np.abs(r))>1e-4)
    logged_u=np.empty(80);logged_u[0]=40
    for t in range(1,80):logged_u[t]=d*logged_u[t-1]-A*logged[t]+B*logged[t-1]+c
    rr=d*logged_u[k-1]-A*logged[k]+B*logged[k-1]+c-logged_u[k]
    check('controller_reads_logged_zero_feasible',np.max(np.abs(rr))<1e-10 and q28_joint(rr,k,A,B,h)['feasible'])
    bad=np.array([.069,.069]);kk=np.array([1,2]);check('perrow_bound_does_not_imply_joint',np.all(abs(bad)<=.07+1e-4) and not q28_joint(bad,kk,1.01,1.,h)['feasible'])
    rng=np.random.default_rng(280028)
    for trial in range(24):
        kk=np.sort(rng.choice(np.arange(1,80),size=30,replace=False));aa=rng.uniform(.4,2.);bb=rng.uniform(.05,.9)*aa;delta=rng.uniform(-h,h,80);rv=(-aa*delta[kk]+bb*delta[kk-1]) if trial%2==0 else rng.normal(scale=.08,size=len(kk));Z=np.zeros((len(kk),80));Z[np.arange(len(kk)),kk]=-aa;Z[np.arange(len(kk)),kk-1]=bb
        oracle=linprog(np.zeros(80),A_ub=np.vstack([Z,-Z]),b_ub=np.r_[rv+1e-4,-rv+1e-4],bounds=[(-h,h)]*80,method='highs',options={'primal_feasibility_tolerance':1e-9})
        check('joint_matches_independent_LP_'+str(trial),q28_joint(rv,kk,aa,bb,h)['feasible']==oracle.success)
        if trial%2==0:check('known_feasible_joint_'+str(trial),oracle.success)
    lattice=q28_lattice(np.delete(np.arange(100)*.07,[20,21,22,65]));check('missing_levels_do_not_widen_cells',abs(lattice['half_width']-.035)<1e-12)
    try:q28_lattice(np.r_[np.arange(99)*.07,7.003])
    except ValueError:checks['nonuniform_lattice_rejected']=True
    else:raise AssertionError('Nonuniform lattice accepted')
    phase=np.zeros(80,dtype=np.int8);f1=q28_prefix(logged_u,logged,phase);changed=logged.copy();changed[f1['k'][f1['cut']-1]+1:]+=2.;f2=q28_prefix(logged_u,changed,phase)
    check('held_pressure_does_not_change_prefix_coefficients',np.array_equal(f1['coef'],f2['coef']))
    check('physical_known_controller_recovered',f1['status']=='physical' and np.allclose(f1['coef'],[d,A,B,c],atol=1e-9,rtol=1e-9))
    shuffled=logged[rng.permutation(80)];rs=d*logged_u[k-1]-A*shuffled[k]+B*shuffled[k-1]+c-logged_u[k]
    check('destructive_pressure_order_rejected',not q28_joint(rs,k,A,B,h)['feasible'])
    return dict(status='passed',checks=checks,check_count=len(checks),synthetic_only=True,approved_csv_reads=0,independent_LP_trials=24)

def run_quantized_controller_v028(root,*,expected_protocol_sha256,expected_source_sha256):
    root=Path(root).resolve();path=root/'research/controller_quantization_protocol_v028.json';q28_need(q28_sha(path)==expected_protocol_sha256,'Protocol changed');plan=json.loads(path.read_text());q28_need(plan['recipe']==V028_RECIPE and plan['status']=='frozen','Unfrozen recipe')
    q28_need(hashlib.sha256(q28_code(root/'research/quantized_controller_components_v028.ipynb').encode()).hexdigest()==expected_source_sha256,'Source changed')
    for n,h in plan['input_hashes'].items():q28_need(q28_sha(root/n)==h,'Input changed: '+n)
    out=root/'outputs/controller_quantization_feasibility_v028';out.mkdir(exist_ok=False);started=time.monotonic();q28_json(out/'preflight.json',q28_preflight())
    with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z:fit=z['A_fit'];val=z['A_validation']
    q28_need(len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size,'Split changed')
    bids,u,p,phase,rc=q28_load_fit(root/'data/train.csv',fit);primary=q28_sample(rc,20262801);replicate=q28_sample(rc,20262802,primary);pool=np.setdiff1d(np.arange(len(bids)),np.r_[primary,replicate]);q28_need(len(pool)==59962 and not np.intersect1d(bids,val).size,'Pool/query contamination')
    q28_json(out/'sample_before_scoring.json',dict(primary_ids=bids[primary].tolist(),replicate_ids=bids[replicate].tolist(),pool_array_sha256=q28_arrayhash(bids[pool]),pool_breaths=len(pool),pool_excludes_both_samples=True,original_Aval_skipped_before_numeric_conversion=True))
    try:lattice=q28_lattice(p[pool][phase[pool]==0])
    except ValueError as e:
        result=dict(status='training_lattice_not_verified',reason=str(e),gate_passed=False,protocol_sha256=expected_protocol_sha256,source_code_sha256=expected_source_sha256,no_pressure_predictor=True,no_A_validation_outcome=True,no_GPU=True,runtime_seconds=time.monotonic()-started)
        q28_json(out/'summary.json',result);return result
    q28_json(out/'lattice.json',lattice);results={};certificates={}
    for label,indices in [('primary',primary),('replicate',replicate)]:
        if label=='replicate' and not results['primary']['gate_passed']:break
        rows=[];cert={}
        for i in indices:
            row,delta=q28_one(u[i],p[i],phase[i],lattice['half_width'],bids[i]);row.update(R=int(rc[i,0]),C=int(rc[i,1]));rows.append(row)
            if delta is not None:cert[str(int(bids[i]))]=delta.tolist()
        pd.DataFrame(rows).to_csv(out/(label+'_rows.csv'),index=False);q28_json(out/(label+'_certificates.json'),cert)
        results[label]=q28_gate(rows,20262803+(label=='replicate'));q28_json(out/(label+'_result.json'),results[label])
    result=dict(status='complete_training_only_conditional_diagnostic',completed_utc=datetime.now(timezone.utc).isoformat(),protocol_sha256=expected_protocol_sha256,source_code_sha256=expected_source_sha256,recipe=V028_RECIPE,lattice=lattice,results=results,gate_passed=bool(len(results)==2 and all(v['gate_passed'] for v in results.values())),replicate_executed='replicate'in results,gradient_model_fits=0,neural_checkpoint_inferences=0,no_A_validation_outcome=True,no_test_data=True,no_GPU=True,no_F_split_or_fit=True,no_submission=True,pressure_predictor_created=False,pressure_recovery_coverage_estimated=False,runtime_seconds=time.monotonic()-started,interpretation='Conditional compatibility using known training pressures. Neither pressure MAE nor input-only reconstruction; unknown gains/reference/offset remain non-identifiable. OLS noisy-regressor bias and gain inflation limit conclusions. A positive result requires a separate input-only test; no automatic GPU promotion.',artifact_sha256={f.name:q28_sha(f) for f in out.iterdir() if f.is_file()})
    q28_json(out/'summary.json',result);print(json.dumps({k:result[k] for k in ['status','results','gate_passed','runtime_seconds']},indent=2));return result
